# Laboratorio: ortogonalidad y Gram–Schmidt

En este laboratorio trabajaremos primero con aritmética simbólica exacta y luego con cálculo numérico. Al finalizar podrá:

1. ejecutar Gram–Schmidt paso a paso;
2. verificar ortogonalidad y ortonormalidad;
3. ortogonalizar polinomios con un producto interno integral;
4. relacionar Gram–Schmidt con la factorización $QR$.

## 1. Herramientas para cálculo exacto

Usaremos `sympy` para conservar fracciones y radicales exactos. Para $u\neq0$,

$$\operatorname{proj}_u(v)=\frac{\langle v,u\rangle}{\langle u,u\rangle}u.$$

In [ ]:
import sympy as sp
import numpy as np
from IPython.display import display, Markdown, Math

sp.init_printing(use_unicode=True)

def producto(u, v):
    return sp.simplify((u.T * v)[0])

def proyeccion(v, u):
    if producto(u, u) == 0:
        raise ValueError('No se puede proyectar sobre el vector cero.')
    return sp.simplify(producto(v, u) / producto(u, u)) * u

def norma(u):
    return sp.sqrt(producto(u, u))

## 2. Ejemplo exacto paso a paso

Partimos de

$$v_1=(1,1,0)^T,\qquad v_2=(1,0,1)^T,\qquad v_3=(0,1,1)^T.$$

Este es el ejemplo desarrollado en las notas y en el Colab original del curso.

In [ ]:
v1 = sp.Matrix([1, 1, 0])
v2 = sp.Matrix([1, 0, 1])
v3 = sp.Matrix([0, 1, 1])

u1 = v1
p21 = proyeccion(v2, u1)
u2 = sp.simplify(v2 - p21)
p31 = proyeccion(v3, u1)
p32 = proyeccion(v3, u2)
u3 = sp.simplify(v3 - p31 - p32)

display(Markdown('### Paso 1'))
display(Math(r'u_1=v_1'))
display(u1)
display(Markdown('### Paso 2'))
display(Math(r'u_2=v_2-\operatorname{proj}_{u_1}(v_2)'))
display(Markdown(r'La proyección es:'))
display(p21)
display(Markdown(r'El segundo vector ortogonal es:'))
display(u2)
display(Markdown('### Paso 3'))
display(Math(r'u_3=v_3-\operatorname{proj}_{u_1}(v_3)-\operatorname{proj}_{u_2}(v_3)'))
display(Markdown(r'Las dos proyecciones son:'))
display(p31, p32)
display(Markdown(r'El tercer vector ortogonal es:'))
display(u3)

### Verificación

La matriz de Gram de una familia $(w_1,\ldots,w_r)$ es

$$G=(\langle w_i,w_j\rangle)_{i,j}.$$

Para una familia ortogonal, $G$ es diagonal; para una familia ortonormal, $G=I$.

In [ ]:
U = sp.Matrix.hstack(u1, u2, u3)
E = sp.Matrix.hstack(*[sp.simplify(u / norma(u)) for u in (u1, u2, u3)])

G_U = sp.simplify(U.T * U)
G_E = sp.simplify(E.T * E)

display(Markdown('**Base ortogonal U:**'))
display(U)
display(Markdown('**Matriz de Gram UᵀU:**'))
display(G_U)
display(Markdown('**Base ortonormal E:**'))
display(E)
display(Markdown('**Comprobación EᵀE:**'))
display(G_E)

assert G_U == sp.diag(2, sp.Rational(3, 2), sp.Rational(4, 3))
assert G_E == sp.eye(3)

## 3. Implementación general exacta

La función siguiente detecta una dependencia lineal: si un nuevo vector pierde todas sus componentes después de restar las proyecciones, el residuo es cero.

In [ ]:
def gram_schmidt_exacto(vectores):
    ortogonales = []
    for k, v in enumerate(vectores, start=1):
        v = sp.Matrix(v)
        u = sp.simplify(v - sum((proyeccion(v, w) for w in ortogonales), sp.zeros(v.rows, 1)))
        if u == sp.zeros(v.rows, 1):
            raise ValueError(f'El vector {k} depende de los anteriores.')
        ortogonales.append(u)
    ortonormales = [sp.simplify(u / norma(u)) for u in ortogonales]
    return ortogonales, ortonormales

U_lista, E_lista = gram_schmidt_exacto([v1, v2, v3])
assert sp.Matrix.hstack(*U_lista) == U
assert sp.simplify(sp.Matrix.hstack(*E_lista).T * sp.Matrix.hstack(*E_lista)) == sp.eye(3)
U_lista, E_lista

## 4. Gram–Schmidt en un espacio de polinomios

En $\mathcal P_2$ usaremos

$$\langle p,q\rangle=\int_{-1}^{1}p(t)q(t)\,dt.$$

Ortogonalizaremos $(1,t,t^2)$. El algoritmo no depende de que los objetos sean columnas de $\mathbb R^n$; depende del producto interno.

In [ ]:
t = sp.symbols('t', real=True)

def producto_polinomial(p, q):
    return sp.integrate(sp.expand(p * q), (t, -1, 1))

def proyeccion_polinomial(p, q):
    return sp.simplify(producto_polinomial(p, q) / producto_polinomial(q, q) * q)

p1 = sp.Integer(1)
p2 = sp.expand(t - proyeccion_polinomial(t, p1))
p3 = sp.expand(t**2 - proyeccion_polinomial(t**2, p1) - proyeccion_polinomial(t**2, p2))
P = [p1, p2, p3]
G_P = sp.Matrix([[producto_polinomial(p, q) for q in P] for p in P])

display(Markdown(r'**Polinomios ortogonales:**'))
display(P)
display(Markdown(r'**Matriz de Gram:**'))
display(G_P)

assert P == [1, t, t**2 - sp.Rational(1, 3)]
assert G_P == sp.diag(2, sp.Rational(2, 3), sp.Rational(8, 45))

## 5. Relación con la factorización QR

Si las columnas de $A$ son linealmente independientes, una factorización reducida $A=QR$ cumple $Q^TQ=I$ y $R$ es triangular superior. Los signos de las columnas de $Q$ no son únicos: cambiar simultáneamente el signo de una columna de $Q$ y de la fila correspondiente de $R$ conserva el producto.

In [ ]:
A = np.array([[1., 1., 0.],
              [1., 0., 1.],
              [0., 1., 1.]])
Q, R = np.linalg.qr(A, mode='reduced')

error_ortonormalidad = np.linalg.norm(Q.T @ Q - np.eye(3))
error_reconstruccion = np.linalg.norm(A - Q @ R)

print('Q =\n', Q)
print('R =\n', R)
print('||Q.T Q - I|| =', error_ortonormalidad)
print('||A - QR|| =', error_reconstruccion)

assert error_ortonormalidad < 1e-12
assert error_reconstruccion < 1e-12

## 6. Columnas casi dependientes

Cuando las columnas son casi dependientes, los errores de redondeo importan. En aplicaciones numéricas se prefiere una rutina estable de QR en lugar de programar literalmente las fórmulas simbólicas.

In [ ]:
eps = 1e-10
A_casi = np.array([[1., 1., 1.],
                   [1., 1. + eps, 1.],
                   [1., 1., 1. + eps]])
Q_casi, R_casi = np.linalg.qr(A_casi, mode='reduced')

print('Condición de A:', np.linalg.cond(A_casi))
print('Error de ortonormalidad:', np.linalg.norm(Q_casi.T @ Q_casi - np.eye(3)))
print('Error de reconstrucción:', np.linalg.norm(A_casi - Q_casi @ R_casi))

assert np.linalg.norm(Q_casi.T @ Q_casi - np.eye(3)) < 1e-12
assert np.linalg.norm(A_casi - Q_casi @ R_casi) < 1e-12

## 7. Ejercicios

1. Aplique `gram_schmidt_exacto` a $(1,0,1)$, $(1,1,0)$ y $(0,1,1)$.
2. Reemplace el tercer vector del ejemplo por la suma de los dos primeros. Explique el mensaje de la función.
3. Normalice los polinomios obtenidos y compruebe que su matriz de Gram es la identidad.
4. Construya una matriz con los vectores del ejercicio 1 como columnas y compare la base ortonormal exacta con la matriz `Q` de NumPy. Recuerde la posible diferencia de signos.